# V5 Auto-Label Training Pipeline
**13 Models | Google Drive Downloads | Crash Protection | Autopilot Ready**

| # | Model | Output Filename | Epochs |
|---|-------|----------------|--------|
| 1-3 | VideoMAE (3 sports) | videomae_{sport}_v5.zip | 20 |
| 4-6 | YOLO Outcome (3 sports) | outcome_classifier_{sport}_v5.pt | 80 |
| 7 | Jersey OCR Universal | jersey_ocr_universal_v5.pt | 120 |
| 8 | Player Detector | player_detector_v5.pt | 100 |
| 9-11 | Ball Detectors (3 sports) | {sport}_ball_detector.pt | 80 |
| 12 | Referee Detector | referee_detector_v5.pt | 80 |
| 13 | Court Zone Detector | basketball_court_zones.pt | 80 |

**All finished .pt files save to:** `Google Drive > MyDrive > clipt_v5_training > downloads/`

### Crash Protection
- Drive backup every 5 epochs + auto-resume on reconnect
- OOM retry with half batch, early stopping
- Just rerun Setup + the crashed cell to continue

### IF COLAB DISCONNECTS
1. Reconnect runtime
2. Rerun **Cell 0** (Setup)
3. Rerun the crashed model cell — auto-resumes from Drive

**Colab Secrets:** `ROBOFLOW_API_KEY` (required), `ANTHROPIC_API_KEY` (optional for auto-labeling)

In [ ]:
#@title **Cell 0: Setup — Run This First** { display-mode: "form" }
!pip install roboflow ultralytics pyyaml anthropic scenedetect[opencv] yt-dlp transformers datasets accelerate -q

from google.colab import userdata, files as colab_files, drive
from roboflow import Roboflow
from ultralytics import YOLO
import os, torch, shutil, glob, yaml, time, traceback, gc, json, cv2
import numpy as np

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/clipt_v5_training'
DRIVE_CHECKPOINTS = f'{DRIVE_ROOT}/checkpoints'
DRIVE_DOWNLOAD = f'{DRIVE_ROOT}/downloads'
DRIVE_DATA = f'{DRIVE_ROOT}/datasets'
DRIVE_MODELS = f'{DRIVE_ROOT}/models'
DRIVE_COSTS = f'{DRIVE_ROOT}/api_costs.json'
for d in [DRIVE_CHECKPOINTS, DRIVE_DOWNLOAD, DRIVE_DATA, DRIVE_MODELS]:
    os.makedirs(d, exist_ok=True)

print(f'Finished models -> {DRIVE_DOWNLOAD}')

# API Keys
try: ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except: ROBOFLOW_API_KEY = ''
try: ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except: ANTHROPIC_API_KEY = ''

key_file = f'{DRIVE_ROOT}/api_keys.json'
if os.path.exists(key_file):
    with open(key_file) as f:
        keys = json.load(f)
        if not ROBOFLOW_API_KEY: ROBOFLOW_API_KEY = keys.get('roboflow', '')
        if not ANTHROPIC_API_KEY: ANTHROPIC_API_KEY = keys.get('anthropic', '')

rf = Roboflow(api_key=ROBOFLOW_API_KEY) if ROBOFLOW_API_KEY else None

assert torch.cuda.is_available(), 'NO GPU'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
HAS_A100 = vram_gb > 40
DEFAULT_BATCH = 16 if HAS_A100 else 8
print(f'GPU: {gpu_name} ({vram_gb:.1f}GB), batch={DEFAULT_BATCH}')

TRAINING_LOG = []

def make_drive_backup_callback(model_name):
    def backup(trainer):
        epoch = trainer.epoch
        if (epoch + 1) % 5 == 0 or epoch == 0:
            dest = f'{DRIVE_CHECKPOINTS}/{model_name}'
            os.makedirs(dest, exist_ok=True)
            for fn in ['weights/best.pt', 'weights/last.pt']:
                src = os.path.join(str(trainer.save_dir), fn)
                if os.path.exists(src):
                    shutil.copy2(src, f'{dest}/{os.path.basename(fn)}')
            print(f'  Drive backup: {model_name} epoch {epoch+1}')
    return backup

def safe_train(model_name, data_path, epochs, imgsz=640, batch=DEFAULT_BATCH, extra_args=None):
    extra_args = extra_args or {}
    if data_path is None:
        TRAINING_LOG.append((model_name, 'SKIPPED', 'No data'))
        print(f'SKIPPED {model_name}'); return None
    data_yaml = data_path
    if hasattr(data_path, 'location'): data_yaml = f'{data_path.location}/data.yaml'
    elif os.path.isdir(str(data_path)): data_yaml = f'{data_path}/data.yaml'
    if not os.path.exists(str(data_yaml)):
        TRAINING_LOG.append((model_name, 'SKIPPED', 'No data.yaml'))
        print(f'SKIPPED {model_name}'); return None
    base = model_name.replace('.pt', '')
    drive_last = f'{DRIVE_CHECKPOINTS}/{base}/last.pt'
    resume = None
    if os.path.exists(drive_last) and os.path.getsize(drive_last) > 1024*1024:
        resume = drive_last; print(f'RESUMING {model_name} from Drive')
    patience = 25 if epochs >= 100 else 15
    for ab in [batch, max(batch//2, 2)]:
        try:
            torch.cuda.empty_cache(); gc.collect()
            model = YOLO(resume) if resume else YOLO('yolov8m.pt')
            model.add_callback('on_train_epoch_end', make_drive_backup_callback(base))
            start = time.time()
            if resume: model.train(resume=True)
            else: model.train(data=data_yaml, epochs=epochs, imgsz=imgsz, batch=ab, name=base, device=0, patience=patience, save_period=5, amp=True, cache=True, **extra_args)
            mins = (time.time()-start)/60
            dest = f'{DRIVE_CHECKPOINTS}/{base}'; os.makedirs(dest, exist_ok=True)
            paths = sorted(glob.glob(f'runs/detect/{base}*/weights/best.pt'))
            if paths: shutil.copy2(paths[-1], f'{dest}/best.pt')
            TRAINING_LOG.append((model_name, 'TRAINED', f'{mins:.1f}min'))
            print(f'{model_name} done in {mins:.1f}min'); return model
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and ab > 2:
                print(f'OOM — retry {model_name} batch={ab//2}')
                torch.cuda.empty_cache(); gc.collect(); resume = None
            else:
                TRAINING_LOG.append((model_name, 'ERROR', str(e)[:60]))
                print(f'ERROR {model_name}: {e}'); return None
    return None

def download_model(model_name, min_map50=0.3):
    base = model_name.replace('.pt', '')
    paths = sorted(glob.glob(f'runs/detect/{base}*/weights/best.pt'))
    path = paths[-1] if paths else None
    if not path:
        db = f'{DRIVE_CHECKPOINTS}/{base}/best.pt'
        if os.path.exists(db): path = db
    if not path:
        TRAINING_LOG.append((model_name, 'MISSING', 'No best.pt'))
        print(f'MISSING: {model_name}'); return False
    sz = os.path.getsize(path)/1024/1024
    dl = f'{DRIVE_DOWNLOAD}/{model_name}'
    shutil.copy2(path, dl)
    print(f'SAVED: {dl} ({sz:.1f}MB)')
    dest = f'{DRIVE_CHECKPOINTS}/{base}'; os.makedirs(dest, exist_ok=True)
    shutil.copy2(path, f'{dest}/best.pt')
    try:
        m = YOLO(path).val(); map50 = m.box.map50
        TRAINING_LOG.append((model_name, 'PASS' if map50>=min_map50 else 'LOW', f'mAP50={map50:.3f} ({sz:.1f}MB)'))
        print(f'  mAP50={map50:.3f}')
    except:
        TRAINING_LOG.append((model_name, 'SAVED', f'{sz:.1f}MB'))
    try: shutil.copy(path, model_name); colab_files.download(model_name)
    except: pass
    return True

def safe_download_dataset(ws, proj, ver, name):
    if not rf: print(f'{name}: No API key'); return None
    for v in ([ver] + [i for i in range(1,6) if i!=ver]):
        try:
            ds = rf.workspace(ws).project(proj).version(v).download('yolov8')
            c = len(glob.glob(f'{ds.location}/train/images/*'))
            print(f'{name} v{v}: {c} images'); return ds
        except: pass
    print(f'{name}: failed'); return None

def track_cost(label, cost):
    costs = {}
    if os.path.exists(DRIVE_COSTS):
        with open(DRIVE_COSTS) as f: costs = json.load(f)
    costs[label] = costs.get(label,0) + cost
    costs['total'] = sum(v for k,v in costs.items() if k!='total')
    with open(DRIVE_COSTS,'w') as f: json.dump(costs, f, indent=2)

print('\nSETUP COMPLETE')

## Section 1: Download All Datasets

In [ ]:
#@title **Cell 1: Download Datasets** { display-mode: "form" }
print('=== Jersey OCR ===')
ds_jersey_1 = safe_download_dataset('footballplayertracking', 'jerseynumberdetectordigitdetector', 6, 'jersey_primary')
ds_jersey_2 = safe_download_dataset('roboflow-100', 'jersey-number-detection-qkeap', 2, 'jersey_rf100')

print('\n=== Players ===')
ds_fb_players = safe_download_dataset('augmented-startups', 'football-player-detection-kucab', 1, 'fb_players')
ds_bb_players = safe_download_dataset('roboflow-universe-projects', 'basketball-players-fy4c2', 1, 'bb_players')
ds_lax_players = safe_download_dataset('ryseai', 'lacrosse-object-detection', 1, 'lax_players')

print('\n=== Balls ===')
ds_bb_ball = safe_download_dataset('roboflow-universe-projects', 'basketball-dataset-o8qt8', 1, 'bb_ball')
ds_fb_ball = safe_download_dataset('roboflow-100', 'american-football-tqfco', 2, 'fb_ball')
ds_lax_ball = safe_download_dataset('ryseai', 'lacrosse-object-detection', 1, 'lax_ball')

print('\n=== Referee ===')
ds_referee = safe_download_dataset('roboflow-universe-projects', 'referee-detection-mstxn', 1, 'referee')

print('\n=== Court/Field ===')
ds_court = safe_download_dataset('computer-vision-d5fjh', 'basketball-detection-dn6fg', 1, 'bb_court')

print('\n=== ALL DATASETS READY ===')

## Section 2: Claude Auto-Labeling (Optional)
Label clips for VideoMAE + outcome classifier training. Skip if you don't have clips.

In [ ]:
#@title **Cell 2A: Download + Split YouTube Videos** { display-mode: "form" }
YOUTUBE_URLS = []  # Add URLs here
SPORT = "basketball"  #@param ["basketball", "football", "lacrosse"]

import yt_dlp
from scenedetect import detect, ContentDetector
import subprocess

CLIPS_DIR = f'/content/clips/{SPORT}'
DL_DIR = f'/content/raw_videos/{SPORT}'
os.makedirs(CLIPS_DIR, exist_ok=True); os.makedirs(DL_DIR, exist_ok=True)

# Restore from Drive
drive_clips = f'{DRIVE_DATA}/clips/{SPORT}'; os.makedirs(drive_clips, exist_ok=True)
for v in glob.glob(f'{drive_clips}/*.mp4'):
    d = f'{CLIPS_DIR}/{os.path.basename(v)}'
    if not os.path.exists(d): shutil.copy2(v, d)

if YOUTUBE_URLS:
    with yt_dlp.YoutubeDL({'format': 'bestvideo[height<=720]+bestaudio/best[height<=720]', 'outtmpl': f'{DL_DIR}/%(id)s.%(ext)s', 'merge_output_format': 'mp4'}) as ydl:
        ydl.download(YOUTUBE_URLS)
    total = 0
    for vp in glob.glob(f'{DL_DIR}/*.mp4'):
        vn = os.path.splitext(os.path.basename(vp))[0]
        scenes = detect(vp, ContentDetector(threshold=27.0))
        cap = cv2.VideoCapture(vp); fps = cap.get(cv2.CAP_PROP_FPS) or 30; n = 0
        for s, e in scenes:
            dur = (e.get_frames()-s.get_frames())/fps
            if 2 <= dur <= 15:
                cp = f'{CLIPS_DIR}/{vn}_clip{n:04d}.mp4'
                subprocess.run(['ffmpeg','-y','-ss',str(s.get_frames()/fps),'-i',vp,'-t',str(dur),'-c:v','libx264','-preset','fast','-crf','23','-an',cp], capture_output=True)
                if os.path.exists(cp) and os.path.getsize(cp)>10000: n+=1; total+=1
        cap.release()
    for c in glob.glob(f'{CLIPS_DIR}/*.mp4'): shutil.copy2(c, f'{drive_clips}/{os.path.basename(c)}')
    print(f'{total} clips extracted')
print(f'Clips ready: {len(glob.glob(f"{CLIPS_DIR}/*.mp4"))}')

In [ ]:
#@title **Cell 2B: Auto-Label with Claude Vision** { display-mode: "form" }
import anthropic, base64

PLAY_TYPES = {
    'basketball': ['layup','jump_shot','dunk','three_pointer','fast_break','rebound','steal','block','assist','free_throw','turnover','other'],
    'football': ['pass_play','run_play','touchdown','interception','sack','field_goal','punt','kickoff','tackle','catch','other'],
    'lacrosse': ['shot','goal','save','ground_ball','face_off','clear','dodge','pass','ride','other'],
}
LABELS_FILE = f'{DRIVE_DATA}/labels/{SPORT}_labels.json'
os.makedirs(f'{DRIVE_DATA}/labels', exist_ok=True)
labels = {}
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE) as f: labels = json.load(f)
    print(f'Loaded {len(labels)} labels from Drive')

clips = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))
unlabeled = [c for c in clips if os.path.basename(c) not in labels]
print(f'Total: {len(clips)}, Labeled: {len(labels)}, To label: {len(unlabeled)}')

if not ANTHROPIC_API_KEY:
    print('Set ANTHROPIC_API_KEY for auto-labeling (optional)')
elif unlabeled:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    types_list = ', '.join(PLAY_TYPES[SPORT]); cost = 0
    for i, cp in enumerate(unlabeled):
        cn = os.path.basename(cp)
        cap = cv2.VideoCapture(cp); tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fb = []
        for frac in [0.1, 0.5, 0.9]:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(tf*frac))
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (512,288))
                _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY,70])
                fb.append(base64.b64encode(buf).decode())
        cap.release()
        if len(fb)<2: labels[cn]='other'; continue
        content = [{"type":"image","source":{"type":"base64","media_type":"image/jpeg","data":b}} for b in fb]
        content.append({"type":"text","text":f"This is a {SPORT} clip. Classify: {types_list}. Reply ONLY the label."})
        try:
            resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=20, messages=[{"role":"user","content":content}])
            label = resp.content[0].text.strip().lower().replace(' ','_')
            if label not in PLAY_TYPES[SPORT]: label = 'other'
            labels[cn] = label
            cost += (resp.usage.input_tokens*0.25+resp.usage.output_tokens*1.25)/1e6
        except: labels[cn] = 'other'
        if (i+1)%10==0:
            with open(LABELS_FILE,'w') as f: json.dump(labels, f, indent=2)
    with open(LABELS_FILE,'w') as f: json.dump(labels, f, indent=2)
    track_cost('claude_labeling', cost)
    print(f'Done! Cost: ${cost:.4f}')

from collections import Counter
if labels:
    for l, c in Counter(labels.values()).most_common(): print(f'  {l}: {c}')

## Section 3: VideoMAE Play Classification
Fine-tune VideoMAE for temporal play classification. One model per sport.

In [ ]:
#@title **Cell 3A: Prepare VideoMAE Datasets** { display-mode: "form" }
from torch.utils.data import Dataset, DataLoader

PLAY_TYPES_ALL = {
    'basketball': ['layup','jump_shot','dunk','three_pointer','fast_break','rebound','steal','block','assist','free_throw','turnover','other'],
    'football': ['pass_play','run_play','touchdown','interception','sack','field_goal','punt','kickoff','tackle','catch','other'],
    'lacrosse': ['shot','goal','save','ground_ball','face_off','clear','dodge','pass','ride','other'],
}

class ClipDataset(Dataset):
    def __init__(self, clips_dir, labels_dict, play_types):
        self.label2id = {t:i for i,t in enumerate(play_types)}
        self.samples = [(cp, self.label2id[labels_dict[os.path.basename(cp)]]) for cp in sorted(glob.glob(f'{clips_dir}/*.mp4')) if os.path.basename(cp) in labels_dict and labels_dict[os.path.basename(cp)] in self.label2id]
        print(f'  {len(self.samples)} samples, {len(play_types)} classes')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, max(total-1,0), 16, dtype=int); frames = []
        for fi in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ret, frame = cap.read()
            if ret: frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224,224)).astype(np.float32)/255.0)
            else: frames.append(np.zeros((224,224,3), dtype=np.float32))
        cap.release()
        return {'pixel_values': torch.tensor(np.transpose(np.stack(frames),(0,3,1,2))), 'labels': torch.tensor(label)}

sport_datasets = {}
for sn in ['basketball','football','lacrosse']:
    lf = f'{DRIVE_DATA}/labels/{sn}_labels.json'
    if os.path.exists(lf):
        with open(lf) as f: sl = json.load(f)
        if sl:
            ds = ClipDataset(f'/content/clips/{sn}', sl, PLAY_TYPES_ALL[sn])
            if len(ds)>=10:
                ts=int(0.8*len(ds)); vs=len(ds)-ts
                t,v = torch.utils.data.random_split(ds,[ts,vs])
                sport_datasets[sn] = (t,v,ds.label2id)
                print(f'  {sn}: {ts} train, {vs} val')
    else: print(f'  {sn}: No labels')
print(f'Ready: {list(sport_datasets.keys())}')

In [ ]:
#@title **Cell 3B: Train VideoMAE + Download** { display-mode: "form" }
from transformers import VideoMAEForVideoClassification, VideoMAEConfig
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import zipfile

for sn, (train_ds, val_ds, label2id) in sport_datasets.items():
    mn = f'videomae_{sn}_v5'
    print(f'\n{"="*50}\nTraining {mn}\n{"="*50}')
    ckpt = f'{DRIVE_CHECKPOINTS}/{mn}/model.pt'; se = 0
    id2label = {v:k for k,v in label2id.items()}
    cfg = VideoMAEConfig.from_pretrained('MCG-NJU/videomae-base', num_labels=len(label2id), label2id=label2id, id2label=id2label)
    model = VideoMAEForVideoClassification.from_pretrained('MCG-NJU/videomae-base', config=cfg, ignore_mismatched_sizes=True).cuda()
    if os.path.exists(ckpt):
        c = torch.load(ckpt); model.load_state_dict(c['model_state']); se = c.get('epoch',0)
        print(f'  Resumed epoch {se}')
    opt = AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
    sched = CosineAnnealingLR(opt, T_max=20)
    tl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
    vl = DataLoader(val_ds, batch_size=4, num_workers=2)
    ba = 0; pc = 0
    for ep in range(se, 20):
        model.train(); tl2=0; co=0; to=0
        for b in tl:
            pv=b['pixel_values'].cuda(); lb=b['labels'].cuda()
            o=model(pixel_values=pv, labels=lb); opt.zero_grad(); o.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            tl2+=o.loss.item(); co+=(o.logits.argmax(-1)==lb).sum().item(); to+=lb.size(0)
        sched.step()
        model.eval(); vc=0; vt=0
        with torch.no_grad():
            for b in vl:
                pv=b['pixel_values'].cuda(); lb=b['labels'].cuda()
                vc+=(model(pixel_values=pv).logits.argmax(-1)==lb).sum().item(); vt+=lb.size(0)
        va = vc/max(vt,1)
        print(f'  Ep {ep+1}/20 loss:{tl2/len(tl):.4f} train:{co/max(to,1):.3f} val:{va:.3f}')
        if va > ba:
            ba=va; pc=0; sd=f'{DRIVE_CHECKPOINTS}/{mn}'; os.makedirs(sd, exist_ok=True)
            torch.save({'model_state':model.state_dict(),'epoch':ep+1,'val_acc':va,'label2id':label2id}, f'{sd}/model.pt')
            model.save_pretrained(f'{DRIVE_MODELS}/{mn}')
            zp = f'{DRIVE_DOWNLOAD}/{mn}.zip'
            with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as zf:
                md2=f'{DRIVE_MODELS}/{mn}'
                for r,d2,fs in os.walk(md2):
                    for fi in fs: fp=os.path.join(r,fi); zf.write(fp, os.path.relpath(fp,md2))
            print(f'  [BEST] saved to Drive')
        else:
            pc+=1
            if pc>=5: print(f'  Early stop ep {ep+1}'); break
    TRAINING_LOG.append((mn, 'TRAINED', f'val_acc={ba:.3f}'))
    torch.cuda.empty_cache()
print('\n=== VideoMAE done ===')

## Section 4: YOLO Outcome Classifiers
Output: `outcome_classifier_{sport}_v5.pt` (matches pipeline expectations)

In [ ]:
#@title **Cell 4: Train Outcome Classifiers** { display-mode: "form" }
# Build classification datasets from labeled clips
for sn in ['basketball','football','lacrosse']:
    lf = f'{DRIVE_DATA}/labels/{sn}_labels.json'
    if not os.path.exists(lf): continue
    with open(lf) as f: sl = json.load(f)
    cd = f'/content/clips/{sn}'; cls_dir = f'/content/yolo_cls/{sn}'
    for cn, label in sl.items():
        cp = f'{cd}/{cn}'
        if not os.path.exists(cp): continue
        cap = cv2.VideoCapture(cp); cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))//2)
        ret, frame = cap.read(); cap.release()
        if not ret: continue
        split = 'train' if hash(cn)%5!=0 else 'val'
        od = f'{cls_dir}/{split}/{label}'; os.makedirs(od, exist_ok=True)
        cv2.imwrite(f'{od}/{cn.replace(".mp4",".jpg")}', cv2.resize(frame,(640,640)))

# Train each
for sn in ['basketball','football','lacrosse']:
    cls_dir = f'/content/yolo_cls/{sn}'
    if not os.path.exists(f'{cls_dir}/train'): print(f'{sn}: No data'); continue
    mn = f'outcome_classifier_{sn}_v5'
    print(f'\nTraining {mn}...')
    safe_train(f'{mn}.pt', cls_dir, epochs=80)
    download_model(f'{mn}.pt')
    torch.cuda.empty_cache()
print('\n=== Outcome classifiers done ===')

## Section 5: Jersey OCR Universal v5
Output: `jersey_ocr_universal_v5.pt`

**Fixes:** 10 digit classes (0-9), proper bboxes, fliplr=0

In [ ]:
#@title **Cell 5A: Prepare OCR Data** { display-mode: "form" }
from PIL import Image, ImageDraw, ImageFont
import random

# Remap Roboflow classes to digits 0-9
def remap_to_digits(ds_dir):
    for ld in glob.glob(f'{ds_dir}/*/labels'):
        dy = os.path.join(os.path.dirname(ld), 'data.yaml'); cm = {}
        if os.path.exists(dy):
            with open(dy) as f: d = yaml.safe_load(f)
            for oid, name in enumerate(d.get('names',[])):
                nc = str(name).strip()
                if nc.isdigit() and 0<=int(nc)<=9: cm[oid]=int(nc)
        if not cm: continue
        for lf in glob.glob(f'{ld}/*.txt'):
            lines = []
            with open(lf) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p)>=5 and int(p[0]) in cm: p[0]=str(cm[int(p[0])]); lines.append(' '.join(p))
            if lines:
                with open(lf,'w') as f: f.write('\n'.join(lines)+'\n')

for ds in [ds_jersey_1, ds_jersey_2]:
    if ds: remap_to_digits(ds.location)

# Generate 2000 synthetic digit images with PROPER bboxes
SYNTH = '/content/synth_ocr'
os.makedirs(f'{SYNTH}/images', exist_ok=True); os.makedirs(f'{SYNTH}/labels', exist_ok=True)
COLORS = [(255,0,0),(0,0,255),(255,255,255),(0,128,0),(255,165,0),(128,0,128),(0,0,0)]
for i in range(2000):
    num = random.randint(0,99); digits = str(num); w,h = 640,640
    bg = random.choice(COLORS); img = Image.new('RGB',(w,h),bg); draw = ImageDraw.Draw(img)
    for _ in range(random.randint(50,200)):
        x1,y1 = random.randint(0,w-1),random.randint(0,h-1)
        nc = tuple(max(0,min(255,c+random.randint(-30,30))) for c in bg)
        draw.rectangle([x1,y1,x1+random.randint(2,8),y1+random.randint(2,8)], fill=nc)
    tc = random.choice([(255,255,255),(0,0,0),(255,255,0)]); fs = random.randint(60,150)
    try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', fs)
    except: font = ImageFont.load_default()
    tw = len(digits)*(fs*0.7); sx = (w-tw)/2+random.randint(-50,50); yp = h/2-fs/2+random.randint(-80,80)
    ll = []
    for j, d in enumerate(digits):
        x = sx+j*(fs*0.7); draw.text((x,yp), d, fill=tc, font=font)
        dw,dh = fs*0.65, fs*1.1
        cx=max(0.01,min(0.99,(x+dw/2)/w)); cy=max(0.01,min(0.99,(yp+dh/2)/h))
        bw=max(0.02,min(0.5,dw/w)); bh=max(0.02,min(0.5,dh/h))
        ll.append(f'{int(d)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    img.save(f'{SYNTH}/images/synth_{i:05d}.jpg')
    with open(f'{SYNTH}/labels/synth_{i:05d}.txt','w') as f: f.write('\n'.join(ll)+'\n')

# Merge all
MERGED = '/content/ocr_merged'
for s in ['train','val']:
    os.makedirs(f'{MERGED}/{s}/images', exist_ok=True); os.makedirs(f'{MERGED}/{s}/labels', exist_ok=True)
for ds in [ds_jersey_1, ds_jersey_2]:
    if not ds: continue
    loc=ds.location; dn=os.path.basename(loc)
    for sp in ['train','valid','val','test']:
        idir=f'{loc}/{sp}/images'; ldir=f'{loc}/{sp}/labels'
        if not os.path.exists(idir): continue
        ts='val' if sp in ['valid','val','test'] else 'train'
        for img in glob.glob(f'{idir}/*'):
            fn=f'{dn}_{os.path.basename(img)}'; shutil.copy2(img, f'{MERGED}/{ts}/images/{fn}')
            lbl=os.path.join(ldir, os.path.splitext(os.path.basename(img))[0]+'.txt')
            if os.path.exists(lbl): shutil.copy2(lbl, f'{MERGED}/{ts}/labels/{os.path.splitext(fn)[0]}.txt')
si=sorted(glob.glob(f'{SYNTH}/images/*.jpg')); sp=int(len(si)*0.8)
for i,ip in enumerate(si):
    fn=os.path.basename(ip); s='train' if i<sp else 'val'
    shutil.copy2(ip, f'{MERGED}/{s}/images/{fn}')
    shutil.copy2(f'{SYNTH}/labels/{fn.replace(".jpg",".txt")}', f'{MERGED}/{s}/labels/{fn.replace(".jpg",".txt")}')

dy = {'path':MERGED,'train':'train/images','val':'val/images','nc':10,'names':{i:str(i) for i in range(10)}}
with open(f'{MERGED}/data.yaml','w') as f: yaml.dump(dy, f)
tc=len(glob.glob(f'{MERGED}/train/images/*')); vc=len(glob.glob(f'{MERGED}/val/images/*'))
print(f'OCR dataset: {tc} train, {vc} val — 10 digit classes')

In [ ]:
#@title **Cell 5B: Train Jersey OCR Universal v5** { display-mode: "form" }
safe_train('jersey_ocr_universal_v5.pt', f'/content/ocr_merged/data.yaml', epochs=120,
    extra_args={'fliplr':0.0, 'mosaic':0.5, 'degrees':5.0, 'scale':0.3})
download_model('jersey_ocr_universal_v5.pt')

## Section 6: Player Detector v5
Output: `player_detector_v5.pt`
FIXED: Uses Roboflow datasets, not empty Claude labels.

In [ ]:
#@title **Cell 6: Train Player Detector v5** { display-mode: "form" }
MP = '/content/player_merged'
for s in ['train','val']:
    os.makedirs(f'{MP}/{s}/images', exist_ok=True); os.makedirs(f'{MP}/{s}/labels', exist_ok=True)
for ds in [ds_fb_players, ds_bb_players, ds_lax_players]:
    if not ds: continue
    loc=ds.location; dn=os.path.basename(loc)
    for sp in ['train','valid','val','test']:
        idir=f'{loc}/{sp}/images'; ldir=f'{loc}/{sp}/labels'
        if not os.path.exists(idir): continue
        ts='val' if sp in ['valid','val','test'] else 'train'
        for img in glob.glob(f'{idir}/*'):
            fn=f'{dn}_{os.path.basename(img)}'; shutil.copy2(img, f'{MP}/{ts}/images/{fn}')
            lbl=os.path.join(ldir, os.path.splitext(os.path.basename(img))[0]+'.txt')
            if os.path.exists(lbl):
                with open(lbl) as f: lines=['0 '+' '.join(l.strip().split()[1:]) for l in f if len(l.strip().split())>=5]
                if lines:
                    with open(f'{MP}/{ts}/labels/{os.path.splitext(fn)[0]}.txt','w') as f: f.write('\n'.join(lines)+'\n')

dy = {'path':MP,'train':'train/images','val':'val/images','nc':1,'names':{0:'player'}}
with open(f'{MP}/data.yaml','w') as f: yaml.dump(dy, f)
tc=len(glob.glob(f'{MP}/train/images/*')); vc=len(glob.glob(f'{MP}/val/images/*'))
print(f'Player dataset: {tc} train, {vc} val')

safe_train('player_detector_v5.pt', f'{MP}/data.yaml', epochs=100)
download_model('player_detector_v5.pt')

## Section 7: Ball Detectors (Per Sport)
Outputs: `basketball_ball_detector.pt`, `football_ball_detector.pt`, `lacrosse_ball_detector.pt`

In [ ]:
#@title **Cell 7: Train Ball Detectors** { display-mode: "form" }
ball_configs = [
    ('basketball_ball_detector', ds_bb_ball),
    ('football_ball_detector', ds_fb_ball),
    ('lacrosse_ball_detector', ds_lax_ball),
]
for mn, ds in ball_configs:
    if not ds: print(f'{mn}: No data'); continue
    loc = ds.location
    for sp in ['train','valid','val','test']:
        ldir = f'{loc}/{sp}/labels'
        if not os.path.exists(ldir): continue
        for lf in glob.glob(f'{ldir}/*.txt'):
            with open(lf) as f: lines = f.readlines()
            remapped = ['0 '+' '.join(l.strip().split()[1:]) for l in lines if len(l.strip().split())>=5]
            if remapped:
                with open(lf,'w') as f: f.write('\n'.join(remapped)+'\n')
    dy_path = f'{loc}/data.yaml'
    with open(dy_path) as f: dy = yaml.safe_load(f)
    dy['nc']=1; dy['names']={0:'ball'}
    with open(dy_path,'w') as f: yaml.dump(dy, f)
    safe_train(f'{mn}.pt', dy_path, epochs=80)
    download_model(f'{mn}.pt')
    torch.cuda.empty_cache()
print('\n=== Ball detectors done ===')

## Section 8: Referee Detector
Output: `referee_detector_v5.pt`

In [ ]:
#@title **Cell 8: Train Referee Detector** { display-mode: "form" }
if ds_referee:
    loc = ds_referee.location
    for sp in ['train','valid','val','test']:
        ldir = f'{loc}/{sp}/labels'
        if not os.path.exists(ldir): continue
        for lf in glob.glob(f'{ldir}/*.txt'):
            with open(lf) as f: lines = f.readlines()
            remapped = ['0 '+' '.join(l.strip().split()[1:]) for l in lines if len(l.strip().split())>=5]
            if remapped:
                with open(lf,'w') as f: f.write('\n'.join(remapped)+'\n')
    dy_path = f'{loc}/data.yaml'
    with open(dy_path) as f: dy = yaml.safe_load(f)
    dy['nc']=1; dy['names']={0:'referee'}
    with open(dy_path,'w') as f: yaml.dump(dy, f)
    safe_train('referee_detector_v5.pt', dy_path, epochs=80)
    download_model('referee_detector_v5.pt')
else: print('No referee dataset')

## Section 9: Court Zone Detector
Output: `basketball_court_zones.pt`

In [ ]:
#@title **Cell 9: Train Court Zone Detector** { display-mode: "form" }
if ds_court:
    safe_train('basketball_court_zones.pt', ds_court, epochs=80)
    download_model('basketball_court_zones.pt')
else: print('No court dataset')
print('\n=== Court zone detector done ===')

## Section 10: Scoreboard Detector (NEW)
Output: `scoreboard_detector_v5.pt`

Detects scoreboard overlay in broadcast footage.
**Why:** Score change = guaranteed highlight. Cross-validates visual detection.

In [ ]:
#@title **Cell 10: Train Scoreboard Detector** { display-mode: "form" }
print('=== Scoreboard Datasets ===')
ds_score_1 = safe_download_dataset('roboflow-universe-projects', 'scoreboard-detection-v2-fwi6p', 1, 'scoreboard_1')
ds_score_2 = safe_download_dataset('sports-analytics-zzpgn', 'scoreboard-detection-v5kaz', 1, 'scoreboard_2')

MS = '/content/scoreboard_merged'
for s in ['train','val']:
    os.makedirs(f'{MS}/{s}/images', exist_ok=True)
    os.makedirs(f'{MS}/{s}/labels', exist_ok=True)

has_data = False
for ds in [ds_score_1, ds_score_2]:
    if not ds: continue
    has_data = True
    loc = ds.location; dn = os.path.basename(loc)
    for sp in ['train','valid','val','test']:
        idir = f'{loc}/{sp}/images'; ldir = f'{loc}/{sp}/labels'
        if not os.path.exists(idir): continue
        ts = 'val' if sp in ['valid','val','test'] else 'train'
        for img in glob.glob(f'{idir}/*'):
            fn = f'{dn}_{os.path.basename(img)}'
            shutil.copy2(img, f'{MS}/{ts}/images/{fn}')
            lbl = os.path.join(ldir, os.path.splitext(os.path.basename(img))[0]+'.txt')
            if os.path.exists(lbl):
                with open(lbl) as f:
                    lines = ['0 '+' '.join(l.strip().split()[1:]) for l in f if len(l.strip().split())>=5]
                if lines:
                    with open(f'{MS}/{ts}/labels/{os.path.splitext(fn)[0]}.txt','w') as f:
                        f.write('\n'.join(lines)+'\n')

if has_data:
    dy = {'path':MS,'train':'train/images','val':'val/images','nc':1,'names':{0:'scoreboard'}}
    with open(f'{MS}/data.yaml','w') as f: yaml.dump(dy, f)
    tc = len(glob.glob(f'{MS}/train/images/*'))
    vc = len(glob.glob(f'{MS}/val/images/*'))
    print(f'Scoreboard dataset: {tc} train, {vc} val')
    safe_train('scoreboard_detector_v5.pt', f'{MS}/data.yaml', epochs=80)
    download_model('scoreboard_detector_v5.pt')
else:
    print('No scoreboard datasets found')

## Section 11: Dead Ball Classifier (NEW)
Output: `dead_ball_classifier_v5.pt`

Binary classifier: live play vs dead time (huddles, timeouts, replays).
**Why:** Skip irrelevant frames, save compute, filter noise from highlights.

In [ ]:
#@title **Cell 11: Train Dead Ball Classifier** { display-mode: "form" }
DB = '/content/dead_ball_cls'
for s in ['train','val']:
    for c in ['live_play','dead_ball']:
        os.makedirs(f'{DB}/{s}/{c}', exist_ok=True)

# Use existing labeled clips: 'other' = dead_ball, everything else = live_play
total_live = 0; total_dead = 0
for sn in ['basketball','football','lacrosse']:
    lf = f'{DRIVE_DATA}/labels/{sn}_labels.json'
    cd = f'/content/clips/{sn}'
    if not os.path.exists(lf): continue
    with open(lf) as f: sl = json.load(f)
    dead_labels = {'other','timeout','huddle','replay','dead_ball'}
    for cn, label in sl.items():
        cp = f'{cd}/{cn}'
        if not os.path.exists(cp): continue
        cap = cv2.VideoCapture(cp)
        tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for frac in [0.2, 0.5, 0.8]:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(tf*frac))
            ret, frame = cap.read()
            if not ret: continue
            cls = 'dead_ball' if label in dead_labels else 'live_play'
            split = 'train' if hash(cn)%5!=0 else 'val'
            fname = f'{sn}_{cn.replace(".mp4","")}_f{int(frac*100)}.jpg'
            cv2.imwrite(f'{DB}/{split}/{cls}/{fname}', cv2.resize(frame,(640,640)))
            if cls == 'live_play': total_live += 1
            else: total_dead += 1
        cap.release()

print(f'Dead ball dataset: {total_live} live, {total_dead} dead')

# If not enough labeled clips, try Claude Vision
if total_live + total_dead < 50 and ANTHROPIC_API_KEY:
    print('Too few clips, using Claude Vision...')
    import anthropic, base64
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    cost = 0
    for vp in glob.glob('/content/raw_videos/*/*.mp4')[:5]:
        cap = cv2.VideoCapture(vp)
        tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for fi in range(0, tf, tf//20):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if not ret: continue
            small = cv2.resize(frame, (512,288))
            _, buf = cv2.imencode('.jpg', small, [cv2.IMWRITE_JPEG_QUALITY,70])
            b64 = base64.b64encode(buf).decode()
            try:
                resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=10,
                    messages=[{"role":"user","content":[
                        {"type":"image","source":{"type":"base64","media_type":"image/jpeg","data":b64}},
                        {"type":"text","text":"Is this live sports action or dead time (huddle/timeout/replay)? Reply ONLY: live or dead"}]}])
                cls = 'dead_ball' if 'dead' in resp.content[0].text.strip().lower() else 'live_play'
                cost += (resp.usage.input_tokens*0.25+resp.usage.output_tokens*1.25)/1e6
            except: cls = 'live_play'
            split = 'train' if hash(fi)%5!=0 else 'val'
            cv2.imwrite(f'{DB}/{split}/{cls}/auto_{os.path.basename(vp)}_{fi}.jpg', cv2.resize(frame,(640,640)))
        cap.release()
    if cost > 0: track_cost('dead_ball_labeling', cost)

tc_l = len(glob.glob(f'{DB}/train/live_play/*'))
tc_d = len(glob.glob(f'{DB}/train/dead_ball/*'))
print(f'Final: {tc_l} live, {tc_d} dead')

if tc_l >= 20 and tc_d >= 20:
    # Use YOLO classify mode
    model = YOLO('yolov8m-cls.pt')
    model.add_callback('on_train_epoch_end', make_drive_backup_callback('dead_ball_classifier_v5'))
    model.train(data=DB, epochs=60, imgsz=640, batch=DEFAULT_BATCH, name='dead_ball_classifier_v5',
                device=0, patience=15, save_period=5, amp=True, cache=True)
    # Save to downloads
    paths = sorted(glob.glob('runs/classify/dead_ball_classifier_v5*/weights/best.pt'))
    if paths:
        shutil.copy2(paths[-1], f'{DRIVE_DOWNLOAD}/dead_ball_classifier_v5.pt')
        print(f'SAVED: {DRIVE_DOWNLOAD}/dead_ball_classifier_v5.pt')
        TRAINING_LOG.append(('dead_ball_classifier_v5.pt', 'TRAINED', 'classifier'))
    torch.cuda.empty_cache()
else:
    print('Not enough data. Add YouTube URLs in Section 2 and rerun.')

## Section 12: Jersey Super-Resolution (NEW)
Output: `jersey_upscaler_v5.pth`

Upscales jersey number crops 4x before OCR.
**Why:** Jersey numbers are often 15-30px. Upscaling to 60-120px gives 20-30% OCR accuracy boost.

In [ ]:
#@title **Cell 12: Train Jersey Super-Resolution** { display-mode: "form" }
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

class SRBlock(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(c,c,3,1,1), nn.BatchNorm2d(c), nn.PReLU(), nn.Conv2d(c,c,3,1,1), nn.BatchNorm2d(c))
    def forward(self, x): return x + self.block(x)

class JerseySR(nn.Module):
    def __init__(self):
        super().__init__()
        self.entry = nn.Sequential(nn.Conv2d(3,64,9,1,4), nn.PReLU())
        self.res = nn.Sequential(*[SRBlock(64) for _ in range(8)])
        self.mid = nn.Sequential(nn.Conv2d(64,64,3,1,1), nn.BatchNorm2d(64))
        self.up = nn.Sequential(
            nn.Conv2d(64,256,3,1,1), nn.PixelShuffle(2), nn.PReLU(),
            nn.Conv2d(64,256,3,1,1), nn.PixelShuffle(2), nn.PReLU())
        self.out = nn.Conv2d(64,3,9,1,4)
    def forward(self, x):
        e = self.entry(x); r = self.mid(self.res(e))
        return torch.clamp(self.out(self.up(e+r)), 0, 1)

class SRDataset(Dataset):
    def __init__(self, img_dir, hr=128):
        self.images = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
        self.hr = hr; self.lr = hr//4
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        hr = T.Compose([T.Resize(self.hr), T.CenterCrop(self.hr), T.ToTensor()])(img)
        lr = T.ToTensor()(T.Resize(self.lr, interpolation=T.InterpolationMode.BICUBIC)(T.ToPILImage()(hr)))
        return lr, hr

# Collect jersey images
sr_dir = '/content/sr_images'
os.makedirs(sr_dir, exist_ok=True)
for img in glob.glob('/content/ocr_merged/train/images/*')[:3000]:
    shutil.copy2(img, f'{sr_dir}/{os.path.basename(img)}')
for img in glob.glob('/content/synth_ocr/images/*'):
    shutil.copy2(img, f'{sr_dir}/{os.path.basename(img)}')

total = len(glob.glob(f'{sr_dir}/*'))
print(f'SR images: {total}')

if total >= 100:
    ds = SRDataset(sr_dir)
    ts = int(0.9*len(ds)); vs = len(ds)-ts
    tds, vds = torch.utils.data.random_split(ds, [ts, vs])
    tl = DataLoader(tds, batch_size=16, shuffle=True, num_workers=2)
    vl = DataLoader(vds, batch_size=16, num_workers=2)

    model = JerseySR().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
    l1 = nn.L1Loss()
    ckpt = f'{DRIVE_CHECKPOINTS}/jersey_upscaler_v5/model.pth'
    os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    best = float('inf')

    if os.path.exists(ckpt):
        c = torch.load(ckpt); model.load_state_dict(c['model_state'])
        best = c.get('val_loss', float('inf'))
        print(f'Resumed, best={best:.4f}')

    for ep in range(30):
        model.train(); tl2 = 0
        for lr, hr in tl:
            lr,hr = lr.cuda(),hr.cuda(); sr = model(lr)
            loss = l1(sr, hr); opt.zero_grad(); loss.backward(); opt.step()
            tl2 += loss.item()
        sched.step()
        model.eval(); vl2 = 0
        with torch.no_grad():
            for lr, hr in vl: lr,hr=lr.cuda(),hr.cuda(); vl2+=l1(model(lr),hr).item()
        t = tl2/len(tl); v = vl2/len(vl)
        print(f'  Ep {ep+1}/30 train={t:.4f} val={v:.4f}')
        if v < best:
            best = v
            torch.save({'model_state':model.state_dict(),'val_loss':v}, ckpt)
            torch.save(model.state_dict(), f'{DRIVE_DOWNLOAD}/jersey_upscaler_v5.pth')
            print(f'  [BEST] saved')

    TRAINING_LOG.append(('jersey_upscaler_v5.pth', 'TRAINED', f'val_loss={best:.4f}'))
    print(f'\nSR model: 32x32 input -> 128x128 output (4x upscale)')
else:
    print('Run OCR section first to generate training images')

## Section 13: Final Report

In [ ]:
#@title **Cell 13: Final Report** { display-mode: "form" }
ALL = [
    'videomae_basketball_v5.zip','videomae_football_v5.zip','videomae_lacrosse_v5.zip',
    'outcome_classifier_basketball_v5.pt','outcome_classifier_football_v5.pt','outcome_classifier_lacrosse_v5.pt',
    'jersey_ocr_universal_v5.pt','player_detector_v5.pt',
    'basketball_ball_detector.pt','football_ball_detector.pt','lacrosse_ball_detector.pt',
    'referee_detector_v5.pt','basketball_court_zones.pt',
    'scoreboard_detector_v5.pt','dead_ball_classifier_v5.pt','jersey_upscaler_v5.pth',
]
print('='*70)
print('  V5 TRAINING — FINAL REPORT (16 MODELS)')
print('='*70)
print(f'\nDRIVE: Google Drive > MyDrive > clipt_v5_training > downloads\n')
found = 0
for mn in ALL:
    dp = f'{DRIVE_DOWNLOAD}/{mn}'
    if os.path.exists(dp):
        sz = os.path.getsize(dp)/1024/1024; print(f'  OK  {mn} ({sz:.1f}MB)'); found+=1
    else: print(f'  --  {mn}')
print(f'\n  {found}/{len(ALL)} models ready')

if os.path.exists(DRIVE_COSTS):
    with open(DRIVE_COSTS) as f: costs = json.load(f)
    print(f'\n  API Costs:')
    for k,v in costs.items(): print(f'    {k}: ${v:.4f}')

if TRAINING_LOG:
    print(f'\nTRAINING LOG:')
    for n,s,d in TRAINING_LOG: print(f'  {s:10s} | {n:45s} | {d}')

print(f'\n{"="*70}')
if found >= 10:
    print('NEXT: Copy all files from Drive downloads -> reelapp/app/model/')
else:
    print(f'{found}/16 done — rerun failed cells')
print('='*70)